# Dự đoán giá iPhone cũ bằng XGBoost

Dataset: `Iphone_Database_filtered_train.csv`  
Mục tiêu: Dự đoán cột **Giá** (VND) từ các đặc trưng của máy.

## 1. Cài thư viện (nếu chưa có)

In [ ]:
# Chạy cell này nếu chưa cài xgboost
# !pip install xgboost scikit-learn pandas matplotlib seaborn

## 2. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')

print(' Import xong!')

## 3. Đọc dữ liệu

In [ ]:
# Đường dẫn file CSV
CSV_PATH = r'output/Iphone_Database_filtered_train.csv'

df = pd.read_csv(CSV_PATH)

print(f' Shape: {df.shape}')
print(f' Cột: {list(df.columns)}')
df.head()

## 4. Khám phá dữ liệu (EDA)

In [ ]:
print('=== Thông tin cơ bản ===')
print(df.dtypes)
print()
print('=== Giá trị null ===')
print(df.isnull().sum())
print()
print('=== Thống kê Giá ===')
print(df['Giá'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Phân phối giá
axes[0].hist(df['Giá'] / 1e6, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Phân phối Giá (triệu VND)', fontsize=13)
axes[0].set_xlabel('Giá (triệu VND)')
axes[0].set_ylabel('Số lượng')

# Giá theo dòng máy (top 15)
top_models = df.groupby('Dòng máy')['Giá'].median().sort_values(ascending=False).head(15)
axes[1].barh(top_models.index, top_models.values / 1e6, color='coral')
axes[1].set_title('Giá trung vị theo Dòng máy (top 15)', fontsize=13)
axes[1].set_xlabel('Giá trung vị (triệu VND)')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Giá theo Tình trạng
df.boxplot(column='Giá', by='Tình trạng', ax=axes[0], grid=False)
axes[0].set_title('Giá theo Tình trạng')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)

# Giá theo Pin_bucket
pin_order = ['Dưới 80%', '80-84%', '85-89%', '90-94%', '95-96%', '97-99%', '100%']
pin_data = [df[df['Pin_bucket'] == p]['Giá'].values for p in pin_order if p in df['Pin_bucket'].values]
pin_labels = [p for p in pin_order if p in df['Pin_bucket'].values]
axes[1].boxplot(pin_data, labels=pin_labels)
axes[1].set_title('Giá theo Pin_bucket')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 5. Chuẩn bị dữ liệu để train

In [ ]:
# Theo hướng dẫn file guide:
# - Dùng Pin_bucket (bỏ Tình trạng pin)
# - Dung lượng (GB) là cột số
# - Các cột còn lại encode category

X = df.drop(columns=['Giá', 'Tình trạng pin'])
y = df['Giá']

numeric_features = ['Dung lượng (GB)']
categorical_features = [col for col in X.columns if col not in numeric_features]

print(f'Features số   : {numeric_features}')
print(f'Features dạng category: {categorical_features}')
print(f'Target: Giá — {y.shape[0]} mẫu')

In [ ]:
# Train / Test split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]} mẫu | Test: {X_test.shape[0]} mẫu')

## 6. Xây dựng Pipeline XGBoost

In [ ]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', 'passthrough', numeric_features),
    ]
)

# XGBoost model (tham số baseline tốt)
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
)

# Pipeline hoàn chỉnh
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

print(' Pipeline XGBoost đã sẵn sàng!')

## 7. Train mô hình

In [ ]:
%%time
pipeline.fit(X_train, y_train)
print('Train xong!')

## 8. Đánh giá mô hình

In [ ]:
y_pred_train = pipeline.predict(X_train)
y_pred_test  = pipeline.predict(X_test)

def print_metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'--- {label} ---')
    print(f'  MAE  : {mae:>15,.0f} VND  (~{mae/1e6:.2f} triệu)')
    print(f'  RMSE : {rmse:>15,.0f} VND  (~{rmse/1e6:.2f} triệu)')
    print(f'  R²   : {r2:.4f}')
    print(f'  MAPE : {mape:.2f}%')

print_metrics(y_train, y_pred_train, 'TRAIN')
print()
print_metrics(y_test, y_pred_test, 'TEST')

In [ ]:
# Cross-validation 5-fold
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
cv_mae = -cv_scores

print('=== KFold Cross-Validation (5-fold) ===')
for i, s in enumerate(cv_mae, 1):
    print(f'  Fold {i}: MAE = {s:,.0f} VND')
print(f'  Mean MAE = {cv_mae.mean():,.0f} VND  (±{cv_mae.std():,.0f})')

## 9. Biểu đồ kết quả

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Actual vs Predicted
ax = axes[0]
ax.scatter(y_test / 1e6, y_pred_test / 1e6, alpha=0.5, color='steelblue', s=20)
lim = max(y_test.max(), y_pred_test.max()) / 1e6
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Dự đoán hoàn hảo')
ax.set_xlabel('Giá thực tế (triệu VND)')
ax.set_ylabel('Giá dự đoán (triệu VND)')
ax.set_title('Actual vs Predicted (Test set)')
ax.legend()

# Phân phối residual
residuals = (y_test - y_pred_test) / 1e6
ax2 = axes[1]
ax2.hist(residuals, bins=40, color='coral', edgecolor='white')
ax2.axvline(0, color='red', linestyle='--')
ax2.set_xlabel('Sai số (triệu VND)  [Actual - Predicted]')
ax2.set_ylabel('Số lượng')
ax2.set_title('Phân phối sai số (Residual)')

plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
# Lấy tên features sau khi encode
ohe_feature_names = pipeline.named_steps['preprocessor'] \
    .named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
all_feature_names = ohe_feature_names + numeric_features

importances = pipeline.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

# Top 20 features quan trọng nhất
top_n = 20
fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(top_n).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Feature Importance (XGBoost)')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()

# Tổng importance theo nhóm cột gốc
print('\n=== Tổng importance theo cột gốc ===')
for col in categorical_features:
    mask = [f.startswith(f'{col}_') for f in feat_imp.index]
    total = feat_imp[mask].sum()
    print(f'  {col:30s}: {total:.4f}')
print(f'  {"Dung lượng (GB)":30s}: {feat_imp["Dung lượng (GB)"]:.4f}')

## 11. Dự đoán với dữ liệu mới

In [ ]:
# Ví dụ predict theo guide
sample = pd.DataFrame([{
    'Dòng máy': 'iPhone 14 Pro Max',
    'Phiên bản': 'Quốc tế (Không khoá mạng)',
    'Tình trạng': 'Đã sử dụng (chưa sửa chữa)',
    'Tinh_trang_tong_hop': 'Zin không trầy xước',
    'Xuất xứ': 'VN/A',
    'Dung lượng (GB)': 256,
    'Pin_bucket': '97-99%',
    'Chính sách bảo hành': 'Hết bảo hành'
}])

predicted = pipeline.predict(sample)[0]
print(f' iPhone 14 Pro Max | 256GB | Zin không trầy | Pin 97-99%')
print(f' Giá dự đoán: {predicted:,.0f} VND  (~{predicted/1e6:.1f} triệu)')

In [ ]:
# Thử thêm vài mẫu khác nhau
samples = pd.DataFrame([
    {
        'Dòng máy': 'iPhone 11',
        'Phiên bản': 'Quốc tế (Không khoá mạng)',
        'Tình trạng': 'Đã sử dụng (chưa sửa chữa)',
        'Tinh_trang_tong_hop': 'Zin không trầy xước',
        'Xuất xứ': 'VN/A',
        'Dung lượng (GB)': 64,
        'Pin_bucket': '90-94%',
        'Chính sách bảo hành': 'Hết bảo hành'
    },
    {
        'Dòng máy': 'iPhone 15 Pro Max',
        'Phiên bản': 'Quốc tế (Không khoá mạng)',
        'Tình trạng': 'Đã sử dụng (chưa sửa chữa)',
        'Tinh_trang_tong_hop': 'Zin không trầy xước',
        'Xuất xứ': 'VN/A',
        'Dung lượng (GB)': 256,
        'Pin_bucket': '97-99%',
        'Chính sách bảo hành': 'Hết bảo hành'
    },
    {
        'Dòng máy': 'iPhone 13',
        'Phiên bản': 'Khóa mạng (Lock)',
        'Tình trạng': 'Đã sửa chữa / thay linh kiện',
        'Tinh_trang_tong_hop': 'Pin thay',
        'Xuất xứ': 'Mỹ (LL/A)',
        'Dung lượng (GB)': 128,
        'Pin_bucket': '100%',
        'Chính sách bảo hành': 'Hết bảo hành'
    },
])

preds = pipeline.predict(samples)
for i, (_, row) in enumerate(samples.iterrows()):
    print(f' {row["Dòng máy"]} | {int(row["Dung lượng (GB)"])}GB | {row["Tình trạng"]}  →  {preds[i]:,.0f} VND')

## 12. Lưu model (tùy chọn)

In [ ]:
import pickle, os

model_path = 'output/xgboost_iphone_price.pkl'
os.makedirs('output', exist_ok=True)

with open(model_path, 'wb') as f:
    pickle.dump(pipeline, f)

print(f'Model đã lưu tại: {model_path}')

# Kiểm tra load lại
with open(model_path, 'rb') as f:
    loaded_pipeline = pickle.load(f)

test_pred = loaded_pipeline.predict(sample)[0]
print(f'Load lại OK — Dự đoán: {test_pred:,.0f} VND')

## 13. (Nâng cao) Tuning tham số XGBoost

In [ ]:
# Chạy cell này nếu muốn tìm tham số tốt hơn (mất ~vài phút)
# from sklearn.model_selection import GridSearchCV
#
# param_grid = {
#     'model__n_estimators': [300, 500, 700],
#     'model__max_depth': [4, 6, 8],
#     'model__learning_rate': [0.03, 0.05, 0.1],
# }
#
# grid_search = GridSearchCV(
#     pipeline, param_grid,
#     cv=3, scoring='neg_mean_absolute_error',
#     n_jobs=-1, verbose=1
# )
# grid_search.fit(X_train, y_train)
# print('Best params:', grid_search.best_params_)
# print('Best MAE:', -grid_search.best_score_)